In [3]:
import polars as pl
import numpy as np

# 1. Carga de 1M de linhas (Equilíbrio entre velocidade e precisão)
print("⚡ Carregando 1M de linhas com Geometria Completa...")
df = pl.read_parquet("statsbomb_completo.parquet", n_rows=1000000)
df = df.rename({col: col.replace(".", "_") for col in df.columns}).sort(["match_id", "index"])

# 2. Coordenadas e Pressão Básica
df = df.with_columns([
    pl.col("location").list.get(0).fill_null(120.0).alias("x"),
    pl.col("location").list.get(1).fill_null(40.0).alias("y"),
    pl.col("under_pressure").fill_null(False).cast(pl.Int8).alias("pressao_binaria")
])

# 3. GEOMETRIA DE ELITE (Aqui entra o Ângulo que faltava!)
x_arr, y_arr = df["x"].to_numpy(), df["y"].to_numpy()
dist = np.sqrt((120 - x_arr)**2 + (40 - y_arr)**2)

# Cálculo do ângulo (Lei dos Cossenos entre a bola e as duas traves)
a = np.sqrt((120 - x_arr)**2 + (36 - y_arr)**2)
b = np.sqrt((120 - x_arr)**2 + (44 - y_arr)**2)
cos_theta = np.clip((a**2 + b**2 - 8**2) / (2 * a * b), -1.0, 1.0)
angulo = np.degrees(np.arccos(cos_theta))

# 4. Injetando tudo no DataFrame
df_contexto = df.with_columns([
    pl.Series("distancia", dist),
    pl.Series("angulo_visao", angulo),
    # Pressão nos últimos 10 lances
    pl.col("pressao_binaria").rolling_mean(window_size=10).over("match_id").alias("pressao_10_lances"),
    # Aceleração: progresso vertical nos últimos 5 lances
    (pl.col("x") - pl.col("x").shift(5).over("match_id")).alias("aceleracao_ataque")
])

# 5. O ALVO (Iminência de gol nos próximos 10 eventos)
df_treino = df_contexto.with_columns([
    (pl.col("type_name") == "Shot").cast(pl.Int8).alias("eh_chute")
]).with_columns([
    pl.col("eh_chute").shift(-10).rolling_max(window_size=10).over("match_id").fill_null(0).alias("iminencia_gol")
])

# Filtro para focar apenas em construção de jogada
df_final = df_treino.filter(
    pl.col("type_name").is_in(["Pass", "Carry", "Dribble"])
).fill_null(0)

print(f"✅ Tudo pronto! Colunas disponíveis: {df_final.columns}")

⚡ Carregando 1M de linhas com Geometria Completa...
✅ Tudo pronto! Colunas disponíveis: ['id', 'index', 'period', 'timestamp', 'minute', 'second', 'possession', 'duration', 'type_id', 'type_name', 'possession_team_id', 'possession_team_name', 'play_pattern_id', 'play_pattern_name', 'team_id', 'team_name', 'tactics_formation', 'tactics_lineup', 'related_events', 'location', 'player_id', 'player_name', 'position_id', 'position_name', 'pass_recipient_id', 'pass_recipient_name', 'pass_length', 'pass_angle', 'pass_height_id', 'pass_height_name', 'pass_end_location', 'pass_body_part_id', 'pass_body_part_name', 'pass_type_id', 'pass_type_name', 'carry_end_location', 'pass_switch', 'pass_outcome_id', 'pass_outcome_name', 'ball_receipt_outcome_id', 'ball_receipt_outcome_name', 'under_pressure', 'duel_type_id', 'duel_type_name', 'pass_aerial_won', 'counterpress', 'interception_outcome_id', 'interception_outcome_name', 'off_camera', 'ball_recovery_recovery_failure', 'pass_assisted_shot_id', 'pass

In [4]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("🚀 Iniciando treinamento do Oráculo...")

df_ml = df_final.to_pandas()

# Lista de ingredientes confirmada
features = [
    'distancia', 'angulo_visao', 'pressao_10_lances', 
    'aceleracao_ataque', 'x', 'y'
]

X = df_ml[features]
y = df_ml['iminencia_gol']

# Cálculo de peso dinâmico (Ouro para AUC alto)
peso_calibrado = len(y[y==0]) / len(y[y==1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_oraculo = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=peso_calibrado, # Agora o modelo vai valorizar o perigo
    tree_method='hist',
    eval_metric='auc'
)

model_oraculo.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# Validação final
preds = model_oraculo.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, preds)

print(f"🔥 SUCESSO! Novo AUC: {auc:.4f}")
model_oraculo.save_model("oraculo_iminencia.json")

🚀 Iniciando treinamento do Oráculo...
🔥 SUCESSO! Novo AUC: 0.7791
